# Soros 13F — factor risk, straight from the Atoti cube in Python

This notebook reproduces every view in the **Soros 13F filings** folder by calling the Atoti cube
**directly from Python** — `cube.query(...)` in the kernel, no FastAPI, no HTTP, no Streamlit.
Each view below is a self-contained, hand-written query feeding a pandas table (grids) or an
altair chart (charts). The view JSON files were the *spec*; the code here is the literal API.

Run order: build the cube once (next cell), then any view cell. Built on free/public data
(SEC 13F + EDGAR XBRL + Stooq), Soros Fund Management's 13F book as a weight overlay.

In [1]:
import datetime as dt
import pandas as pd
import altair as alt
import notebook_helpers as N

session, cube = N.build()                       # builds the six-frame cube on :9096 (~1-2 min)
h, l, m = cube.hierarchies, cube.levels, cube.measures
D = dt.date(2024, 12, 31)                        # latest monthly COB in the sample; all views as-of D
SOROS = l["Book"] == "Soros"
print("cube ready — hierarchies:", sorted(n for _, n in h))

Welcome to Atoti 0.9.15!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


cube built: 1,044,329 leaf rows, 10 style factors, 7 scenario sets
cube ready — hierarchies: ['Book', 'Date', 'FactorDim', 'ScenarioDay', 'ScenarioSet', 'Security']


## Direct cube access — the two idioms

Everything below is built from two `cube.query` shapes: **(1)** scalar measures grouped by a
level under a `filter`, and **(2)** the synthetic **`ScenarioDay`** dimension, which unpacks a
scenario P&L *vector* into one row per day — so a vector measure becomes an ordinary group-by.

In [2]:
# (1) scalar measures, grouped by ScenarioSet, sliced to Soros @ latest COB — "the switch":
#     the same measures, every scenario mode, side by side.
display(cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]], filter=SOROS & (l["Date"] == D)))

# (2) ScenarioDay unpacks the COVID P&L vector into a per-day series (head):
display(cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020")).head())

,ScenarioSet,Scenario VaR 99,Scenario worst loss,Total VaR 99
0,Evt:COVID2020,0.112319,0.132158,0.112545
1,Evt:Rates2022,0.042824,0.050167,0.043414
2,Evt:Selloff2018,0.030556,0.030931,0.031377
3,HistFull,0.032777,0.132158,0.033544
4,Hypo:MomentumCrash,0.002472,0.002472,0.007549
5,Hypo:RiskOff,-0.001773,-0.001773,0.007349
6,Hypo:ValueRotation,0.001243,0.001243,0.00724


,ScenarioDay,Scenario PnL at day,Scenario date at day (epoch)
0,0,0.009578,18295
1,1,0.016265,18296
2,2,0.012595,18297
3,3,0.001344,18298
4,4,-0.007132,18299


## L1 · Book risk summary

In [3]:
# ── L1 · Book risk summary ────────────────────────────────────────────────────────────────
# The top of the drill-down: one row, the whole book. Soros's 1-day 99% VaR right now, split into
# the factor-driven tail (Scenario VaR 99, historical simulation) and the idiosyncratic tail
# (Specific vol), then combined (Total VaR 99). Everything below decomposes THIS number.
# Direct API: no group-by beyond Book; slice to Soros / latest COB / the full-history set.
df = cube.query(
    m["Total VaR 99"], m["Scenario VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Book"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
)
N.style_grid(df)

,Book,Total VaR 99,Scenario VaR 99,Specific vol
0,Soros,3.354%,3.278%,0.307%


## L2 · Factor contributions

In [4]:
# ── L2 · Factor contributions ─────────────────────────────────────────────────────────────
# Drill the book VaR onto the risk factors. Marginal Scenario VaR 99 is each factor's P&L on the
# book's 1%-tail scenario (ADDITIVE — the factors sum to the book factor-VaR); % of Scenario VaR
# 99 is its share. "Which factors own the tail?" — here Market dominates a long-equity book.
# Direct API: group by Factor; same Soros / COB / HistFull slice; sorted biggest-first.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
0,Market,Market,3.136%,95.569%
5,Style,Liquidity,0.152%,4.620%
9,Style,Size,0.032%,0.975%
8,Style,ResidVol,0.020%,0.594%
10,Style,Value,0.013%,0.401%
3,Style,Growth,0.005%,0.165%
7,Style,NonLinSize,0.003%,0.102%
2,Style,EarnYield,0.003%,0.085%
4,Style,Leverage,-0.003%,-0.100%
6,Style,Momentum,-0.035%,-1.074%


## L2 · Factor incremental VaR

In [5]:
# ── L2 · Factor incremental VaR ───────────────────────────────────────────────────────────
# The diversification-aware companion to contributions: Incremental Scenario VaR 99 is how much
# book VaR is RELEASED if a factor's exposure is removed (the book tail recomputed without it).
# Unlike the marginals it is NOT additive (VaR is sub-additive) — "what does cutting this factor
# actually buy me?". Direct API: group by Factor; sort by the incremental column.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["Incremental Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,Incremental Scenario VaR 99
0,Market,Market,3.136%,2.772%
8,Style,ResidVol,0.020%,0.020%
10,Style,Value,0.013%,0.013%
5,Style,Liquidity,0.152%,0.006%
3,Style,Growth,0.005%,0.005%
7,Style,NonLinSize,0.003%,0.003%
2,Style,EarnYield,0.003%,0.003%
4,Style,Leverage,-0.003%,-0.003%
6,Style,Momentum,-0.035%,-0.035%
1,Style,Beta,-0.044%,-0.044%


## L2 · Issuer contributions

In [6]:
# ── L2 · Issuer contributions ─────────────────────────────────────────────────────────────
# Same decomposition, now by name. Marginal Total VaR 99 is each issuer's additive share of the
# COMBINED tail (factor + specific in quadrature — the Euler split), meaningful per-name where
# idiosyncratic risk lives. "Which positions carry the book's risk?". Direct API: group by Issuer
# (the cube returns the Country/Sector/Issuer security path); top names first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Issuer,Marginal Total VaR 99,% of Total VaR 99
6,US,Consumer Discretionary,"JD.com, Inc.",0.241%,7.182%
54,US,Industrials,Alibaba Group Holding Ltd,0.233%,6.934%
74,US,Information Technology,Alphabet Inc.,0.172%,5.123%
100,US,Utilities,GFL Environmental Inc.,0.130%,3.880%
0,US,Communication Services,EchoStar CORP,0.121%,3.609%
2,US,Consumer Discretionary,AMAZON COM INC,0.117%,3.498%
1,US,Communication Services,Liberty Broadband Corp,0.117%,3.486%
84,US,Information Technology,"Salesforce, Inc.",0.102%,3.041%
73,US,Industrials,"Uber Technologies, Inc",0.099%,2.956%
67,US,Industrials,JACOBS SOLUTIONS INC.,0.080%,2.394%


## L2 · Issuer incremental VaR

In [7]:
# ── L2 · Issuer incremental VaR ───────────────────────────────────────────────────────────
# Per-name diversification view: Incremental Total VaR 99 = book Total VaR released by removing
# the name (factor tail re-struck on the reduced book + its specific variance stripped). The cut
# list a PM reads to de-risk. Direct API: group by Issuer; sort by the incremental column.
df = cube.query(
    m["Marginal Total VaR 99"], m["Incremental Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Issuer,Marginal Total VaR 99,Incremental Total VaR 99
6,US,Consumer Discretionary,"JD.com, Inc.",0.241%,0.220%
54,US,Industrials,Alibaba Group Holding Ltd,0.233%,0.220%
74,US,Information Technology,Alphabet Inc.,0.172%,0.169%
100,US,Utilities,GFL Environmental Inc.,0.130%,0.130%
1,US,Communication Services,Liberty Broadband Corp,0.117%,0.117%
2,US,Consumer Discretionary,AMAZON COM INC,0.117%,0.116%
0,US,Communication Services,EchoStar CORP,0.121%,0.113%
84,US,Information Technology,"Salesforce, Inc.",0.102%,0.101%
73,US,Industrials,"Uber Technologies, Inc",0.099%,0.097%
67,US,Industrials,JACOBS SOLUTIONS INC.,0.080%,0.079%


## L3 · FactorGroup contributions

In [8]:
# ── L3 · FactorGroup contributions ────────────────────────────────────────────────────────
# Roll the factor contributions up one level — Market vs Style — for the one-line "is this a
# market bet or a style bet?" read. Same additive Marginal Scenario VaR 99 / share, grouped at
# the FactorGroup level of the FactorDim hierarchy. Direct API: levels=[FactorGroup].
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["FactorGroup"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Marginal Scenario VaR 99,% of Scenario VaR 99
0,Market,3.136%,95.569%
1,Style,0.145%,4.431%


## L3 · Sector contributions

In [9]:
# ── L3 · Sector contributions ─────────────────────────────────────────────────────────────
# The book's risk by GICS sector — additive Marginal Total VaR 99 per sector (factor + specific),
# the cross-sectional concentration view. Group at the Sector level of the Security hierarchy (the
# cube returns Country/Sector). Direct API: levels=[Sector]; biggest contributors first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Sector"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df)

,Country,Sector,Marginal Total VaR 99,% of Total VaR 99
6,US,Industrials,0.960%,28.586%
7,US,Information Technology,0.647%,19.267%
1,US,Consumer Discretionary,0.408%,12.150%
4,US,Financials,0.363%,10.810%
5,US,Health Care,0.355%,10.581%
0,US,Communication Services,0.238%,7.095%
10,US,Utilities,0.151%,4.499%
8,US,Materials,0.118%,3.516%
9,US,Real Estate,0.059%,1.766%
3,US,Energy,0.035%,1.049%


## Stress · Factor contributions (COVID 2020)

In [10]:
# ── Stress · Factor contributions (COVID 2020) ────────────────────────────────────────────
# The L2 factor decomposition re-struck under a STRESS set instead of the full history: the ONLY
# change from "L2 · Factor contributions" is the ScenarioSet slice (Evt:COVID2020). That one
# switch — slicing the ScenarioSet hierarchy — turns historical-sim VaR into event-replay stress
# VaR, and Market's tail balloons. Direct API: identical query, ScenarioSet == 'Evt:COVID2020'.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
0,Market,Market,13.561%,102.611%
6,Style,Momentum,0.187%,1.414%
5,Style,Liquidity,0.090%,0.684%
10,Style,Value,0.060%,0.453%
2,Style,EarnYield,0.012%,0.089%
3,Style,Growth,-0.001%,-0.006%
8,Style,ResidVol,-0.008%,-0.057%
7,Style,NonLinSize,-0.036%,-0.275%
1,Style,Beta,-0.051%,-0.389%
4,Style,Leverage,-0.066%,-0.501%


## VaR trend (HistFull)

In [11]:
# ── VaR trend (HistFull) ──────────────────────────────────────────────────────────────────
# The book's risk THROUGH TIME: 99% factor VaR, total VaR, and specific vol at every monthly COB
# (the full 2016-2024 calendar, historical-sim set). One line per measure — "is the book getting
# riskier?". Direct API: group by Date (no Date filter -> the whole series); melt the three
# measure columns to long form for a colour-per-measure line in altair.
trend = cube.query(
    m["Scenario VaR 99"], m["Total VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Date"]],
    filter=SOROS & (l["ScenarioSet"] == "HistFull"),
).reset_index().sort_values("Date")
meas = ["Scenario VaR 99", "Total VaR 99", "Specific vol"]
trend["Date"] = pd.to_datetime(trend["Date"])
trend[meas] = trend[meas].astype(float)
long = trend.melt(id_vars="Date", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_line(point=True).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["Date:T", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=380, width="container", title="VaR trend — Soros (HistFull)")

alt.Chart(...)

## Stress board — all scenarios

In [12]:
# ── Stress board · all scenarios ──────────────────────────────────────────────────────────
# Every scenario set side by side at the latest COB: factor VaR, worst single-day loss, and total
# VaR per set — historical-sim vs the three event replays vs the hypotheticals. The CRO's "how bad
# across regimes" board. Direct API: group by ScenarioSet (no ScenarioSet filter -> all sets);
# melt to long for grouped horizontal bars, worst-first.
board = cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=SOROS & (l["Date"] == D),
).reset_index()
meas = ["Scenario VaR 99", "Scenario worst loss", "Total VaR 99"]
board[meas] = board[meas].astype(float)
long = board.melt(id_vars="ScenarioSet", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_bar().encode(
    y=alt.Y("ScenarioSet:N", title=None, sort="-x"),
    x=alt.X("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    yOffset=alt.YOffset("Measure:N"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["ScenarioSet:N", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=420, width="container", title="Stress board — Soros @ 2024-12-31")

alt.Chart(...)

## Scenario P&L — COVID 2020

Two structurally-**different** graphs, built the explicit way: **two `cube.query` calls → two
DataFrames → two graphs**, one graph per DataFrame. The query (a tabular dataset) is the primitive;
the graph just references it. The scenario engine produces a P&L *vector* over the COVID replay
window; the synthetic `ScenarioDay` dimension unpacks it. **Query 1** is the book path
(`levels=[ScenarioDay]`); **Query 2** adds a grain (`levels=[ScenarioDay, Sector]`) and is
re-ordered worst→best into a loss curve — the ordering computed **in the DataFrame**, not the chart.

In [13]:
# ── COVID 2020 · QUERY 1 of 2 -> the book P&L path DataFrame ───────────────────────────────
# Two different graphs need two different queries -> two DataFrames. This is query 1: the per-day
# BOOK path. ScenarioDay unpacks the scenario P&L vector into one row per day WITHOUT exploding the
# cube, so the path is a plain group-by: levels=[ScenarioDay]. The 99% VaR threshold and the worst-
# loss day ride along as book-level marker measures. Epoch-day ints -> real dates in pandas.
covid = SOROS & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020")
covid_path = cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    m["Scenario VaR line at day"], m["Scenario worst pnl at day"],
    m["Scenario worst date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"]], filter=covid,
).reset_index()
for c in ["Scenario PnL at day", "Scenario VaR line at day", "Scenario worst pnl at day"]:
    covid_path[c] = covid_path[c].astype(float)
covid_path["date"]       = pd.to_datetime(covid_path["Scenario date at day (epoch)"].astype("int64"), unit="D")
covid_path["worst_date"] = pd.to_datetime(covid_path["Scenario worst date at day (epoch)"].astype("int64"), unit="D")
covid_path        # DataFrame 1 — the tabular dataset graph 1 draws from

,index,ScenarioDay,Scenario PnL at day,Scenario date at day (epoch),Scenario VaR line at day,Scenario worst pnl at day,Scenario worst date at day (epoch),date,worst_date
0,0,0,0.009578,18295,-0.112319,-0.132158,18337,2020-02-03,2020-03-16
1,1,1,0.016265,18296,-0.112319,-0.132158,18337,2020-02-04,2020-03-16
2,2,2,0.012595,18297,-0.112319,-0.132158,18337,2020-02-05,2020-03-16
3,3,3,0.001344,18298,-0.112319,-0.132158,18337,2020-02-06,2020-03-16
4,4,4,-0.007132,18299,-0.112319,-0.132158,18337,2020-02-07,2020-03-16
...,...,...,...,...,...,...,...,...,...
77,77,77,0.002191,18404,-0.112319,-0.132158,18337,2020-05-22,2020-03-16
78,78,78,0.025853,18408,-0.112319,-0.132158,18337,2020-05-26,2020-03-16
79,79,79,0.020521,18409,-0.112319,-0.132158,18337,2020-05-27,2020-03-16
80,80,80,-0.010863,18410,-0.112319,-0.132158,18337,2020-05-28,2020-03-16


In [14]:
# ── COVID 2020 · GRAPH 1 — book scenario P&L path (draws from `covid_path`) ────────────────
# The graph just references DataFrame 1: a P&L line + zero baseline + the red 99% VaR rule and the
# red worst-loss point (book-level markers, collapsed to one mark each with aggregate:min).
base     = alt.Chart(covid_path)
zero     = base.mark_rule(color="#d8d5cd").encode(y=alt.datum(0))
line     = base.mark_line(point=True, color="#2a4d69", strokeWidth=1.3).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("Scenario PnL at day:Q", axis=alt.Axis(format="%"), title="scenario P&L"),
    tooltip=[alt.Tooltip("date:T"),
             alt.Tooltip("Scenario PnL at day:Q", format=".2%", title="P&L")])
var_rule = base.mark_rule(color="#c0392b", strokeDash=[4, 4]).encode(
    y=alt.Y("min(Scenario VaR line at day):Q"))                      # one book VaR-99 rule
worst    = base.mark_point(color="#c0392b", size=80, filled=True).encode(
    x=alt.X("min(worst_date):T"), y=alt.Y("min(Scenario worst pnl at day):Q"),
    tooltip=[alt.Tooltip("min(worst_date):T", title="worst date"),
             alt.Tooltip("min(Scenario worst pnl at day):Q", format=".2%", title="worst P&L")])

(zero + line + var_rule + worst).properties(
    height=300, width="container", title="COVID 2020 — book scenario P&L path")

alt.LayerChart(...)

In [15]:
# ── COVID 2020 · QUERY 2 of 2 -> the by-Sector loss-curve DataFrame ────────────────────────
# Query 2: the SAME per-day P&L at a finer grain -- levels=[ScenarioDay, Sector] -- so each day
# splits into its sector contributions. The worst->best ordering is computed HERE, in the DataFrame:
# roll the sectors back up to a book total per day, sort those day-totals ascending (worst/most-
# negative first), carry the `rank` onto every sector row, then physically sort the frame. (Graph 2
# then uses sort=None to honour this order.) Tick labels stay dates (%d %b); the order is the rank.
covid_sector = cube.query(
    m["Scenario PnL at day"], m["Scenario date at day (epoch)"],
    mode="raw", levels=[l["ScenarioDay"], l["Sector"]], filter=covid,
).reset_index()
covid_sector["Scenario PnL at day"] = covid_sector["Scenario PnL at day"].astype(float)
covid_sector["date"] = pd.to_datetime(covid_sector["Scenario date at day (epoch)"].astype("int64"), unit="D")

order = (covid_sector.groupby("ScenarioDay", as_index=False)["Scenario PnL at day"]
                     .sum().sort_values("Scenario PnL at day"))       # worst (most negative) first
order["rank"] = range(len(order))
covid_sector = (covid_sector.merge(order[["ScenarioDay", "rank"]], on="ScenarioDay")
                            .sort_values(["rank", "Sector"]))         # df now in worst->best order
covid_sector["label"] = covid_sector["date"].dt.strftime("%d %b")
covid_sector      # DataFrame 2 — the tabular dataset graph 2 draws from

,index,ScenarioDay,Country,Sector,Scenario PnL at day,Scenario date at day (epoch),date,rank,label
319,319,29,US,Communication Services,-0.009376,18337,2020-03-16,0,16 Mar
320,320,29,US,Consumer Discretionary,-0.012433,18337,2020-03-16,0,16 Mar
321,321,29,US,Consumer Staples,-0.000879,18337,2020-03-16,0,16 Mar
322,322,29,US,Energy,-0.002048,18337,2020-03-16,0,16 Mar
323,323,29,US,Financials,-0.019179,18337,2020-03-16,0,16 Mar
...,...,...,...,...,...,...,...,...,...
391,391,35,US,Industrials,0.032134,18345,2020-03-24,81,24 Mar
392,392,35,US,Information Technology,0.018905,18345,2020-03-24,81,24 Mar
393,393,35,US,Materials,0.004358,18345,2020-03-24,81,24 Mar
394,394,35,US,Real Estate,0.002608,18345,2020-03-24,81,24 Mar


In [16]:
# ── COVID 2020 · GRAPH 2 — scenario P&L by Sector, the loss curve (draws from `covid_sector`) ─
# The graph just references DataFrame 2: sectors stacked to the book total, x in the DataFrame's own
# worst->best row order (sort=None), labelled by date.
alt.Chart(covid_sector).mark_area(opacity=0.85, line={"strokeWidth": 0.4}).encode(
    x=alt.X("label:N", sort=None, title="scenario date (worst → best)",   # sort=None => keep df order
            axis=alt.Axis(labelAngle=-45, labelOverlap=True)),
    y=alt.Y("Scenario PnL at day:Q", stack="zero", axis=alt.Axis(format="%"), title="scenario P&L"),
    color=alt.Color("Sector:N", scale=alt.Scale(scheme="set2"),
                    legend=alt.Legend(orient="top", title=None)),
    tooltip=[alt.Tooltip("date:T", title="date"), "Sector:N",
             alt.Tooltip("Scenario PnL at day:Q", format=".2%", title="P&L")],
).properties(height=300, width="container", title="COVID 2020 — scenario P&L by Sector")

alt.Chart(...)